In [1]:
!pip install yfinance newspaper3k vaderSentiment fredapi textblob scikit-learn xgboost lightgbm plotly dash requests beautifulsoup4 pandas-datareader

^C


In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import requests
from bs4 import BeautifulSoup
import newspaper
from newspaper import Article
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textblob import TextBlob
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import xgboost as xgb
import lightgbm as lgb

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.offline as pyo

import pickle
import json
import time
from fredapi import Fred
import pandas_datareader.data as web

ModuleNotFoundError: No module named 'newspaper'

In [ ]:
class SamsungNewsCollector:
    def __init__(self):
        self.analyzer = SentimentIntensityAnalyzer()
        self.sources = [
            'https://finance.yahoo.com/quote/005930.KS/news',
            'https://www.reuters.com/companies/005930.KS',
            'https://www.bloomberg.com/quote/005930:KS'
        ]

    def collect_news_data(self, days_back=30):
        news_data = []
        end_date = datetime.now()
        start_date = end_date - timedelta(days=days_back)

        # Yahoo Finance News
        try:
            ticker = yf.Ticker("005930.KS")
            news = ticker.news
            for article in news[:20]:
                news_data.append({
                    'date': datetime.fromtimestamp(article['providerPublishTime']),
                    'title': article['title'],
                    'source': 'Yahoo Finance',
                    'url': article['link']
                })
        except:
            pass

        # Web scraping for additional sources
        keywords = ['Samsung Electronics', 'Samsung', '삼성전자']
        for keyword in keywords:
            try:
                search_url = f"https://news.google.com/rss/search?q={keyword}&hl=en-US&gl=US&ceid=US:en"
                response = requests.get(search_url)
                soup = BeautifulSoup(response.content, 'xml')
                items = soup.find_all('item')[:10]

                for item in items:
                    pub_date = pd.to_datetime(item.pubDate.text)
                    if start_date <= pub_date <= end_date:
                        news_data.append({
                            'date': pub_date,
                            'title': item.title.text,
                            'source': 'Google News',
                            'url': item.link.text
                        })
            except:
                continue

        return pd.DataFrame(news_data)

    def analyze_sentiment(self, news_df):
        sentiments = []

        for _, row in news_df.iterrows():
            text = row['title']

            # VADER sentiment
            vader_scores = self.analyzer.polarity_scores(text)

            # TextBlob sentiment
            blob = TextBlob(text)
            textblob_polarity = blob.sentiment.polarity
            textblob_subjectivity = blob.sentiment.subjectivity

            sentiments.append({
                'date': row['date'],
                'vader_compound': vader_scores['compound'],
                'vader_positive': vader_scores['pos'],
                'vader_negative': vader_scores['neg'],
                'vader_neutral': vader_scores['neu'],
                'textblob_polarity': textblob_polarity,
                'textblob_subjectivity': textblob_subjectivity
            })

        sentiment_df = pd.DataFrame(sentiments)

        # Aggregate daily sentiment scores
        daily_sentiment = sentiment_df.groupby(sentiment_df['date'].dt.date).agg({
            'vader_compound': ['mean', 'std', 'count'],
            'vader_positive': 'mean',
            'vader_negative': 'mean',
            'textblob_polarity': ['mean', 'std'],
            'textblob_subjectivity': 'mean'
        }).round(4)

        daily_sentiment.columns = ['_'.join(col).strip() for col in daily_sentiment.columns]
        daily_sentiment = daily_sentiment.reset_index()
        daily_sentiment['date'] = pd.to_datetime(daily_sentiment['date'])

        return daily

In [ ]:
class EconomicIndicatorsCollector:
    def __init__(self):
        self.fred = Fred(api_key='your_fred_api_key')  # Replace with actual API key

    def collect_economic_data(self, start_date, end_date):
        economic_data = {}

        # Key economic indicators
        indicators = {
            'USD_KRW': 'DEXKOUS',  # USD/KRW exchange rate
            'VIX': 'VIXCLS',       # Volatility Index
            'DJI': 'DJIA',         # Dow Jones Industrial Average
            'NASDAQ': 'NASDAQCOM', # NASDAQ Composite
            'KOSPI': None,         # Will use yfinance for this
            'US_10Y': 'GS10',      # 10-Year Treasury Rate
            'OIL': 'DCOILWTICO'    # Oil prices
        }

        for name, series_id in indicators.items():
            try:
                if series_id:
                    data = self.fred.get_series(series_id, start=start_date, end=end_date)
                    economic_data[name] = data
                elif name == 'KOSPI':
                    kospi = yf.download('^KS11', start=start_date, end=end_date)['Close']
                    economic_data[name] = kospi
            except:
                continue

        # Additional market data using yfinance
        try:
            # Semiconductor ETF
            semiconductor_etf = yf.download('SMH', start=start_date, end=end_date)['Close']
            economic_data['SEMICONDUCTOR_ETF'] = semiconductor_etf

            # Technology sector
            tech_etf = yf.download('XLK', start=start_date, end=end_date)['Close']
            economic_data['TECH_ETF'] = tech_etf

        except:
            pass

        # Convert to DataFrame
        econ_df = pd.DataFrame(economic_data)